In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# -----------------------------------------------------------------------------
# 超参数设置（适配你的RTX 3050 Ti，调小了batch和序列长度）
# -----------------------------------------------------------------------------
BATCH_SIZE = 8       # 批量大小 B
SEQ_LEN = 50         # 序列长度 T
HIDDEN_DIM = 128     # 隐藏维度 D
ACTION_DIM = 10      # 离散动作空间维度
LR = 3e-4            # 学习率
BETA = 0.1           # KL惩罚系数（控制策略更新幅度）
GAMMA = 0.99         # 折扣因子
LAMBDA = 0.95        # GAE的λ系数
CLIP_EPS = 0.2       # PPO的clip系数

# -----------------------------------------------------------------------------
# 1. 对应架构图：Policy Model（Actor，黄色高亮）
# 输入: q: 状态/观测序列, shape=(B, T, D)
# 输出: o: 策略模型的隐藏表示, shape=(B, T, D)
#       logits: 动作分布的logits, shape=(B, T, ACTION_DIM)
# -----------------------------------------------------------------------------
class PolicyModel(nn.Module):
    def __init__(self, hidden_dim=HIDDEN_DIM, action_dim=ACTION_DIM):
        super().__init__()
        # 简化MLP示例，实际可换成Transformer/LSTM/LLM backbone
        self.backbone = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.action_head = nn.Linear(hidden_dim, action_dim)

    def forward(self, q):
        # q: (B, T, D) 输入状态/观测
        batch_size, seq_len, _ = q.shape
        # 共享处理每个时间步
        x = self.backbone(q)  # (B, T, D) → 对应架构图中的输出o
        logits = self.action_head(x)  # (B, T, ACTION_DIM) 动作分布logits
        return x, logits  # x=o, logits用于计算分布和KL

# -----------------------------------------------------------------------------
# 2. 对应架构图：Reference Model（参考/旧策略，浅蓝色）
# 输入: o: 策略模型的隐藏表示, shape=(B, T, D)
# 输出: ref_logits: 参考策略的动作分布logits, shape=(B, T, ACTION_DIM)
# 作用: 计算与当前策略的KL散度，作为更新约束
# -----------------------------------------------------------------------------
class ReferenceModel(nn.Module):
    def __init__(self, hidden_dim=HIDDEN_DIM, action_dim=ACTION_DIM):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.action_head = nn.Linear(hidden_dim, action_dim)
        # 冻结参数：训练时不更新参考模型
        for param in self.parameters():
            param.requires_grad = False

    def forward(self, o):
        # o: (B, T, D) 来自Policy Model的输出
        x = self.backbone(o)  # (B, T, D)
        ref_logits = self.action_head(x)  # (B, T, ACTION_DIM)
        return ref_logits

# -----------------------------------------------------------------------------
# 3. 对应架构图：Reward Model（浅蓝色）
# 输入: o: 策略模型的隐藏表示, shape=(B, T, D)
# 输出: r: 每个时间步的奖励信号, shape=(B, T)
# -----------------------------------------------------------------------------
class RewardModel(nn.Module):
    def __init__(self, hidden_dim=HIDDEN_DIM):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)  # 输出标量奖励
        )

    def forward(self, o):
        # o: (B, T, D)
        logits = self.backbone(o)  # (B, T, 1)
        r = logits.squeeze(-1)  # (B, T) 去除最后一维，得到奖励序列
        return r

# -----------------------------------------------------------------------------
# 4. 对应架构图：Value Model（Critic，黄色高亮）
# 输入: o: 策略模型的隐藏表示, shape=(B, T, D)
# 输出: v: 每个时间步的状态价值估计, shape=(B, T)
# -----------------------------------------------------------------------------
class ValueModel(nn.Module):
    def __init__(self, hidden_dim=HIDDEN_DIM):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)  # 输出标量价值
        )

    def forward(self, o):
        # o: (B, T, D)
        logits = self.backbone(o)  # (B, T, 1)
        v = logits.squeeze(-1)  # (B, T) 去除最后一维，得到价值序列
        return v

In [ ]:
# -----------------------------------------------------------------------------
# 5. 对应架构图：GAE（广义优势估计）模块
# 输入: rewards: 修正后的奖励, shape=(B, T)
#       values: 价值模型输出, shape=(B, T)
# 输出: advantages: 优势函数A, shape=(B, T)
#       returns: 回报序列（用于价值训练）, shape=(B, T)
# -----------------------------------------------------------------------------
def compute_gae(rewards, values, gamma=GAMMA, lam=LAMBDA):
    advantages = torch.zeros_like(rewards)
    returns = torch.zeros_like(rewards)
    next_value = 0.0  # 终止状态价值，这里简化为0

    # 从后往前计算（GAE标准实现）
    for t in reversed(range(rewards.size(1))):  # t从T-1到0
        # 时序差分误差 δ_t = r_t + γ*v_{t+1} - v_t
        delta = rewards[:, t] + gamma * next_value - values[:, t]
        # 优势更新 A_t = δ_t + γ*λ*A_{t+1}
        advantages[:, t] = delta + gamma * lam * (advantages[:, t+1] if t+1 < rewards.size(1) else 0.0)
        # 回报更新 R_t = r_t + γ*R_{t+1}
        returns[:, t] = rewards[:, t] + gamma * (returns[:, t+1] if t+1 < rewards.size(1) else 0.0)
        # 更新next_value为当前价值
        next_value = values[:, t]

    return advantages, returns

In [ ]:

# -----------------------------------------------------------------------------
# 6. PPO损失函数（包含策略损失、价值损失、熵损失）
# -----------------------------------------------------------------------------
def ppo_loss(policy_logits, ref_logits, advantages, returns, values,
             clip_eps=CLIP_EPS, value_coef=0.5, entropy_coef=0.01):
    # 1. 计算当前策略与参考策略的概率比
    policy_dist = torch.distributions.Categorical(logits=policy_logits)
    ref_dist = torch.distributions.Categorical(logits=ref_logits)
    actions = policy_dist.sample()  # (B, T) 采样动作
    log_probs = policy_dist.log_prob(actions)  # (B, T) 当前策略的对数概率
    ref_log_probs = ref_dist.log_prob(actions)  # (B, T) 参考策略的对数概率
    ratio = torch.exp(log_probs - ref_log_probs)  # (B, T) 概率比 r_t = exp(logp - logp_old)

    # 2. 计算clip策略损失（PPO核心）
    surr1 = ratio * advantages  # (B, T)
    surr2 = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps) * advantages  # (B, T)
    policy_loss = -torch.min(surr1, surr2).mean()  # 最大化目标 → 最小化负目标

    # 3. 计算价值损失（MSE）
    value_loss = F.mse_loss(values, returns)

    # 4. 计算熵损失（鼓励探索）
    entropy = policy_dist.entropy().mean()
    entropy_loss = -entropy  # 最大化熵 → 最小化负熵

    # 5. 总损失
    total_loss = policy_loss + value_coef * value_loss + entropy_coef * entropy_loss

    return total_loss, policy_loss, value_loss, entropy_loss

In [ ]:

# -----------------------------------------------------------------------------
# 7. 单步训练流程（完全对应架构图数据流）
# -----------------------------------------------------------------------------
def train_step(q, policy_model, ref_model, reward_model, value_model, optimizer):
    # 1. Policy Model 前向：输入q → 输出o和policy_logits
    o, policy_logits = policy_model(q)  # o=(B,T,D), policy_logits=(B,T,ACTION_DIM)

    # 2. 并行前向：Reference/Reward/Value模型输入o
    # 2.1 Reference Model：计算ref_logits和KL散度
    ref_logits = ref_model(o)  # (B,T,ACTION_DIM)
    policy_dist = torch.distributions.Categorical(logits=policy_logits)
    ref_dist = torch.distributions.Categorical(logits=ref_logits)
    kl_div = torch.distributions.kl_divergence(policy_dist, ref_dist).mean(dim=-1)  # (B,T) 每个时间步的KL

    # 2.2 Reward Model：计算原始奖励r
    raw_rewards = reward_model(o)  # (B,T)

    # 2.3 Value Model：计算状态价值v
    values = value_model(o)  # (B,T)

    # 3. 对应架构图中的⊕操作：融合奖励 = 原始奖励 - β*KL惩罚
    modified_rewards = raw_rewards - BETA * kl_div  # (B,T)

    # 4. GAE计算优势A和回报
    advantages, returns = compute_gae(modified_rewards, values)  # 都是(B,T)

    # 5. 计算PPO损失
    total_loss, policy_loss, value_loss, entropy_loss = ppo_loss(
        policy_logits=policy_logits,
        ref_logits=ref_logits,
        advantages=advantages,
        returns=returns,
        values=values
    )

    # 6. 反向传播和优化
    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()

    # 7. 同步参考模型（每隔N步复制Policy参数，这里简化为每次同步）
    ref_model.load_state_dict(policy_model.state_dict())

    # 返回损失
    return {
        "total_loss": total_loss.item(),
        "policy_loss": policy_loss.item(),
        "value_loss": value_loss.item(),
        "entropy_loss": entropy_loss.item(),
        "mean_kl": kl_div.mean().item()
    }

In [ ]:

# -----------------------------------------------------------------------------
# 示例运行
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    # 初始化模型
    policy = PolicyModel()
    ref = ReferenceModel()
    reward = RewardModel()
    value = ValueModel()

    # 优化器：只更新Policy和Value，Ref冻结
    optimizer = optim.Adam(
        list(policy.parameters()) + list(value.parameters()),
        lr=LR
    )

    # 示例输入：随机生成的状态序列q
    q = torch.randn(BATCH_SIZE, SEQ_LEN, HIDDEN_DIM)  # (B, T, D)

    # 执行一次训练步骤
    loss_dict = train_step(q, policy, ref, reward, value, optimizer)

    # 打印损失
    print("训练损失:")
    for k, v in loss_dict.items():
        print(f"{k}: {v:.4f}")